# Step 02 — Feature Engineering

This notebook explores the engineered feature matrix produced by `pipelines/02_features.py`.
Features are derived from the raw macro series through:

1. Cross-asset ratios (10 derived columns)
2. Log transforms
3. Column selection (`initial_features`)
4. Bernstein polynomial gap-filling + Taylor extrapolation
5. Smoothed first/second/third derivatives via `np.gradient`
6. Final column selection (`clustering_features`)

**Run `python pipelines/02_features.py` before executing this notebook.**

## Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

import logging

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import DATA_DIR
from market_regime import plotting

setup_logging("INFO")
log = logging.getLogger("02_features")

cfg = load()

run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

print("RunConfig:", run_cfg)
print("DATA_DIR: ", DATA_DIR)

## Load Feature Matrix

The processed feature matrix is stored at `data/processed/features.parquet`.
Each row is a quarter; columns include raw series, log-transforms, and derivatives.

In [ ]:
FEATURES_PATH = DATA_DIR / "processed" / "features.parquet"

try:
    features = pd.read_parquet(FEATURES_PATH)
    log.info("Loaded features: shape=%s", features.shape)
except FileNotFoundError:
    print(f"ERROR: Feature file not found at {FEATURES_PATH}")
    print("Run 'python pipelines/02_features.py' first.")
    features = None
except Exception as exc:
    print(f"ERROR loading features: {exc}")
    features = None

## Column Groups by Prefix

Features are organized by prefix:
- **Raw / derived**: `sp500`, `credit_spread`, `div_yield2`, etc.
- **Log-transformed**: columns beginning with `log_`
- **First derivative**: columns ending with `_d1`
- **Second derivative**: columns ending with `_d2`
- **Third derivative**: columns ending with `_d3`

In [ ]:
if features is not None:
    cols = features.columns.tolist()

    log_cols = [c for c in cols if c.startswith("log_")]
    d1_cols  = [c for c in cols if c.endswith("_d1")]
    d2_cols  = [c for c in cols if c.endswith("_d2")]
    d3_cols  = [c for c in cols if c.endswith("_d3")]
    raw_cols = [c for c in cols if c not in log_cols + d1_cols + d2_cols + d3_cols]

    print(f"Shape: {features.shape[0]} quarters x {features.shape[1]} columns")
    print(f"Date range: {features.index.min().date()} to {features.index.max().date()}")
    print()
    print(f"Raw / derived columns  : {len(raw_cols):3d}")
    print(f"Log-transformed (log_) : {len(log_cols):3d}")
    print(f"First derivative (_d1) : {len(d1_cols):3d}")
    print(f"Second derivative (_d2): {len(d2_cols):3d}")
    print(f"Third derivative (_d3) : {len(d3_cols):3d}")
    print(f"Total                  : {len(cols):3d}")
    print()
    print("Raw/derived sample:", raw_cols[:8])
    print("Log sample:",         log_cols[:8])
    print("d1 sample:",          d1_cols[:8])

## Feature Distributions

Histograms for a subset of features. Higher-order derivatives (`_d2`, `_d3`) are
excluded by default to keep the grid readable. Useful for spotting outliers or
heavily skewed distributions before clustering.

In [ ]:
if features is not None:
    plotting.plot_feature_distributions(features, run_cfg)

## Feature Correlations

Correlation heatmap for the top-40 most-variable features.
Highly correlated clusters confirm expected economic relationships
(e.g., equity valuation metrics move together).

In [ ]:
if features is not None:
    plotting.plot_feature_correlations(features, run_cfg, top_n=40)

## NaN Counts per Column

After Bernstein polynomial gap-filling, most interior NaNs should be resolved.
Remaining NaNs typically appear at the leading or trailing edges of the date range
where a series was not yet available.

In [ ]:
if features is not None:
    nan_counts = features.isna().sum().sort_values(ascending=False)
    non_zero = nan_counts[nan_counts > 0]
    if non_zero.empty:
        print("No NaNs found in feature matrix — gap filling was complete.")
    else:
        print(f"Columns with NaN values ({len(non_zero)} of {len(features.columns)}):")
        display(non_zero.rename("nan_count").to_frame())

## Describe Statistics — Top 20 by Std

Summary statistics sorted by standard deviation. The most variable features
are often the most informative for clustering.

In [ ]:
if features is not None:
    desc = features.describe().T.sort_values("std", ascending=False)
    print("Top 20 features by standard deviation:")
    display(desc.head(20))